# Omniverse Crop Intelligence - Farm Twin Demo

**GoMyCode x NVIDIA hackathon, 2026-09-27.** This notebook builds the entire
agricultural digital twin **headless**: no GPU, no Omniverse install, no server.

What you will see, in order:
1. The full headless test suite (18 tests) proving the OpenUSD invariants.
2. The layer stack being generated: `farm_twin.usda` + `RUNTIME > SIM > DATA > ASSET`.
3. The layer separation itself - definitions and live values are different files.
4. A 30-day FarmWise-style season replayed as **USD time samples** into the
   RUNTIME layer, then scrubbed day-by-day (day 42 vs 55 vs 67).
5. A farm map + curves plot, and (if `scan.ply` was uploaded) the 3DGS scan
   composing into the same stage.

> Honesty first: what is real vs synthetic is tabulated at the bottom and in
> `docs/04_AUDIT_AND_PIVOT.md` section 10. Soil chemistry = AgriFusion schema,
> NDVI/GDD/weather = FarmWise schema, values = synthetic demo seeds; no IoT,
> no soil moisture, no SUBSTOR/SimCast maths - those are Phase 2 (roadmap).


## 0. Get the repo
`REPO_URL` points at the team GitHub repo. If it is private, upload the repo
zip to the Colab file browser instead and set `USE_UPLOAD = True`.


In [ ]:
REPO_URL = "https://github.com/Only1allan/3d_crop_Intelligence-v0.git"
BRANCH = "main"
USE_UPLOAD = False

import os, sys, subprocess, shutil
WORK = "/content/twin"
if USE_UPLOAD:
    WORK = "/content/repo_uploaded"  # adjust to wherever the zip was extracted
elif not os.path.isdir(WORK):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, os.path.join(WORK, "exts", "omniverse_crop_intelligence"))
print("repo at", WORK)


## 1. Install the pinned toolchain
`usd-core==26.8` is the demo freeze (2026-07-20 release, wheels for py3.9-3.14).


In [ ]:
!pip install -q -r requirements-dev.txt
import importlib.metadata as md
print("usd-core", md.version("usd-core"))


## 2. Headless invariant suite
These 18 tests enforce the architecture: root layer purity, layer order,
definitions-vs-values separation, RuntimeWriter firewall, replay semantics,
and the 3DGS-to-USD scan conversion.


In [ ]:
import subprocess
res = subprocess.run([sys.executable, "-m", "pytest",
                      "exts/omniverse_crop_intelligence/tests/", "-q"],
                     capture_output=True, text=True)
print(res.stdout[-1200:])
assert res.returncode == 0, "test suite must be green"


## 3. Build the farm layer stack (pure CPU)


In [ ]:
res = subprocess.run([sys.executable, "tools/build_layers_headless.py",
                      "--out", "demo_layers", "--demo", "--overwrite"],
                     capture_output=True, text=True)
print(res.stdout)
assert res.returncode == 0


### What was just created
| file | role | strength |
|---|---|---|
| `farm_twin.usda` | root: ONLY the sublayer list | - |
| `RUNTIME_LYR.usda` | live telemetry + time-sampled replay | strongest |
| `SIM_LYR.usda` | reserved for physics (Phase 3) | |
| `DATA_LYR.usda` | zones + sensor identity/placement | |
| `ASS_LYR.usda` | 3D scan reference + placeholder ground | weakest |

Root layer purity is not a convention - it is tested and enforced. Live
values can be wiped (`clear_runtime_layer`) without ever touching geometry.


In [ ]:
for name in ("farm_twin.usda", "RUNTIME_LYR.usda"):
    print(f"===== demo_layers/{name} =====")
    print(open(f"demo_layers/{name}").read()[:1600])
    print()


## 4. Replay a 30-day season as USD time samples
The FarmWise `DailySnapshot`-style series (`notebooks/data/farmwise_replay.csv`,
DAP 38-67, Nyandarua-plausible synthetic values) is authored through the same
`RuntimeWriter` Phase 2 ingestion will use - straight into `RUNTIME_LYR` as
time samples. No stage reload, no new prims, definitions untouched.


In [ ]:
import csv
from pxr import Usd
from omniverse_crop_intelligence import settings as S
from omniverse_crop_intelligence.schema import FarmZoneAPI
from omniverse_crop_intelligence.sensor_manager import SensorManager
from omniverse_crop_intelligence.phase2.runtime_writer import RuntimeWriter
from omniverse_crop_intelligence.stage_builder import StageBuilder

stack = StageBuilder("demo_layers").build(overwrite=False)
writer = RuntimeWriter(stack)
rows = list(csv.DictReader(open("notebooks/data/farmwise_replay.csv")))
days = [int(r["dap"]) for r in rows]

n_zone = writer.write_zone_series("Zone_DemoPlot", {
    "ndvi": [(d, float(r["ndvi"])) for d, r in zip(days, rows)],
    "accumulatedGDD": [(d, float(r["gdd_cum"])) for d, r in zip(days, rows)],
    "daysAfterPlanting": [(d, float(d)) for d in days],
})
n_weather = writer.write_sensor_series("WeatherStation_C", {
    "ambientTemperature": [(d, float(r["temp_c"])) for d, r in zip(days, rows)],
    "relativeHumidity": [(d, float(r["rh_pct"])) for d, r in zip(days, rows)],
    "precipitation": [(d, float(r["precip_mm"])) for d, r in zip(days, rows)],
})
# Final-day phenology default (dap 67 lands in the TuberBulking window 46-80).
writer.write_zone("Zone_DemoPlot", {"growthStage": "tuber_bulking", "daysAfterPlanting": 67.0})
StageBuilder.save_all(stack)
print(f"time samples written -> zone: {n_zone}, weather sensor: {n_weather}")


## 5. Scrub the season: same stage, three days
Composed values at day 42 / 55 / 67 - the RUNTIME time samples resolve
per-timecode exactly like a 4D digital twin should.


In [ ]:
zone_api = FarmZoneAPI(stack.stage.GetPrimAtPath(f"{S.ZONES_PATH}/Zone_DemoPlot"))
sensor_api = SensorManager(stack).get_sensor("WeatherStation_C")

print(f"{'dap':>4} {'ndvi':>6} {'GDD':>7} {'tempC':>6} {'RH%':>5} {'rain':>5}")
for d in (42, 55, 67):
    t = Usd.TimeCode(float(d))
    print(f"{d:>4} {zone_api.get('ndvi', t):>6.3f} {zone_api.get('accumulatedGDD', t):>7.1f}"
          f" {sensor_api.get('ambientTemperature', t):>6.1f} {sensor_api.get('relativeHumidity', t):>5.1f}"
          f" {sensor_api.get('precipitation', t):>5.1f}")
print("growth stage (default = final day):", zone_api.get("growthStage"))


## 6. Farm map + season curves


In [ ]:
import matplotlib.pyplot as plt
from pxr import UsdGeom
from omniverse_crop_intelligence.schema import FarmSensorAPI
from pxr import UsdGeom
from omniverse_crop_intelligence import settings as S

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))

for prim in SensorManager(stack).list_sensors():
    sapi = FarmSensorAPI(prim)
    t = UsdGeom.Xformable(prim).GetLocalTransformation().ExtractTranslation()
    c = S.KIND_COLORS.get(sapi.get("kind"), (0.5, 0.5, 0.5))
    ax1.scatter(t[0], t[1], s=160, color=c, edgecolor="k", zorder=3)
    ax1.annotate(sapi.get("sensorId"), (t[0], t[1]), textcoords="offset points", xytext=(8, 5), fontsize=8)
zx = UsdGeom.Xformable(stack.stage.GetPrimAtPath(f"{S.ZONES_PATH}/Zone_DemoPlot")).GetLocalTransformation().ExtractTranslation()
ax1.scatter(zx[0], zx[1], marker="s", s=140, color="#7a5c2e", edgecolor="k", zorder=2, label="Zone_DemoPlot (Shangi, 1 ha)")
ax1.set_title("Farm stage: sensors + zone (metres, Z-up)")
ax1.set_xlabel("x [m]"); ax1.set_ylabel("y [m]"); ax1.legend(fontsize=8); ax1.set_aspect("equal")

dap = [int(r["dap"]) for r in rows]
ax2.plot(dap, [float(r["ndvi"]) for r in rows], "o-", ms=3, label="NDVI", color="#2e7d32")
ax2.plot(dap, [float(r["gdd_cum"])/100.0 for r in rows], "s--", ms=3, label="GDD / 100", color="#ef6c00")
ax2.plot(dap, [float(r["rh_pct"])/10.0 for r in rows], "^:", ms=3, label="RH % / 10", color="#1565c0")
ax2.axvspan(50, 52, color="0.85", label="dry spell")
ax2.axvspan(47, 48, color="#bbdefb", label="rain event")
ax2.set_title("30-day season replay (synthetic FarmWise-style series)")
ax2.set_xlabel("days after planting"); ax2.legend(fontsize=8)
plt.tight_layout()
import os; os.makedirs("notebooks/out", exist_ok=True)
plt.savefig("notebooks/out/farm_map.png", dpi=140)
plt.show()


## 7. (Optional) Compose the 3DGS scan into the twin
Run `gs_scan.ipynb` first, download its `scan.ply`, and upload it here - or
skip: the twin composes either way (PlaceholderGround covers the unresolved
reference by design).


In [ ]:
import os
if os.path.exists("scan.ply"):
    from omniverse_crop_intelligence.scan_io import convert
    print(convert("scan.ply", "demo_layers/farm_scan.usda"))
    stack = StageBuilder("demo_layers").build(overwrite=False)
    scan = stack.stage.GetPrimAtPath(S.FARM_SCAN_PATH)
    pc = scan.GetChild("PointCloud")
    print("FarmScan children:", [c.GetName() for c in scan.GetChildren()])
    from pxr import UsdGeom
    print("points:", len(UsdGeom.Points(pc).GetPointsAttr().Get()) if pc else 0)
else:
    print("scan.ply not found - skipping. The twin still composes (placeholder ground).")


## What this proves (claims ledger)

| demo segment | claim | class |
|---|---|---|
| test suite | 18 headless invariants green on `usd-core==26.8` | REAL |
| layer stack | root purity + RUNTIME>SIM>DATA>ASSET separation | REAL, tested |
| replay | 90+ time samples through the enforced write path, no reload | REAL mechanism |
| scrub | composed values resolve per timecode (4D) | REAL |
| soil chemistry | AgriFusion schema (pH, EC, OC, N, P, K) | REAL schema, synthetic values |
| NDVI/GDD/weather | FarmWise schema | REAL schema, synthetic values |
| soil moisture / hourly RH / SRAD | schema supports them; no legacy source - Phase 2 IoT | ROADMAP |
| SUBSTOR / SimCast | driver mapping done; models = external Phase 2 services | ROADMAP |

Next: `gs_scan.ipynb` (GPU 3D reconstruction on a free T4) and the browser
splat viewer. Full record: `docs/04_AUDIT_AND_PIVOT.md`.
